<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">  <tr style="border: none;">    <td style="vertical-align: middle; border: none; padding: 15px 20px;">      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">        2.3.3 Estudios de Caso: Visualización de Datos Numéricos 🧪      </h1>      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">        Especialización en Ciencia de Datos | Visual Analytics and Critical Thinking      </p>      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">        Universidad Santo Tomás — Seccional Tunja      </p>    </td>    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">        Módulo 05      </span><br>      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>    </td>  </tr></table><div align="center" style="margin-top: 15px; margin-bottom: 15px;">  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Visual%20Analytics%20and%20Critical%20Thinking/05%20-%20Visualizacion%20de%20Datos%20Numericos/03_Estudios_de_Caso_Numericos.ipynb" target="_parent">    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>  </a></div>

---## Objetivos de Aprendizaje 🔎1. Aplicar de forma integrada las técnicas del notebook anterior (line graphs, histogramas, scatter plots, box plots) a un **caso realista con datos sintéticos**.2. Practicar el flujo completo: generar datos, explorar su distribución, detectar anomalías y comunicar hallazgos.3. Interpretar visualizaciones numéricas en un contexto de negocio concreto: ventas mensuales y monitoreo de un sistema.

## Recursos Recomendados 📚- Few, S. (2009). *Now You See It: Simple Visualization Techniques for Quantitative Analysis*.- Documentación de [pandas: resample](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html) para series temporales.

---## 1. Planteamiento del Caso 📋**Escenario:** Eres el analista de datos de una empresa de comercio electrónico. El equipo directivo te pide dos análisis:1. **Analizar las ventas mensuales** de los últimos 3 años para detectar tendencia y estacionalidad.2. **Monitorear el tiempo de respuesta** (en milisegundos) del sistema de checkout, para saber si cumple el acuerdo de servicio (SLA) de 300 ms.Ambos son problemas de **datos numéricos**: ventas mensuales es una serie continua ordenada en el tiempo; tiempo de respuesta es una variable continua medida en miles de transacciones.

In [ ]:
# ============================================================# Generación de datos sintéticos para el caso# ============================================================import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsnp.random.seed(2024)# --- Dataset 1: Ventas mensuales (3 años, con tendencia + estacionalidad) ---meses = pd.date_range('2022-01-01', periods=36, freq='MS')tendencia = np.linspace(80, 140, 36)estacionalidad = 15 * np.sin(2 * np.pi * np.arange(36) / 12 - np.pi / 2) + 15ruido = np.random.normal(0, 5, 36)ventas = tendencia + estacionalidad + ruidodf_ventas = pd.DataFrame({'mes': meses, 'ventas_millones': ventas})# --- Dataset 2: Tiempos de respuesta del sistema (10,000 transacciones) ---tiempos_normales = np.random.gamma(shape=2.0, scale=90, size=9700)  # mstiempos_lentos = np.random.gamma(shape=2.0, scale=350, size=300)    # incidentes de lentitudtiempo_respuesta = np.concatenate([tiempos_normales, tiempos_lentos])np.random.shuffle(tiempo_respuesta)df_latencia = pd.DataFrame({'tiempo_respuesta_ms': tiempo_respuesta})# --- Dataset 3: Tráfico diario vs tiempo de respuesta promedio (para el scatter) ---trafico_diario = np.random.uniform(500, 5000, 90)latencia_promedio = 60 + 0.045 * trafico_diario + np.random.normal(0, 15, 90)df_trafico = pd.DataFrame({'trafico_diario': trafico_diario, 'latencia_promedio_ms': latencia_promedio})print("Dataset 1 — Ventas mensuales:")print(df_ventas.head())print(f"\nDataset 2 — Tiempos de respuesta: {len(df_latencia):,} transacciones")print(df_latencia.describe().round(1))print(f"\nDataset 3 — Tráfico vs latencia: {len(df_trafico)} días")

---## 2. Técnica 1 — Line Graph: Tendencia de Ventas Mensuales 📈Usamos un gráfico de línea porque el eje X (mes) tiene un **orden temporal natural**, lo cual permite visualizar tanto la tendencia general como la estacionalidad.

In [ ]:
# ============================================================# Técnica 1: Line graph de ventas mensuales# ============================================================fig, ax = plt.subplots(figsize=(11, 5))ax.plot(df_ventas['mes'], df_ventas['ventas_millones'], marker='o',        color='#1e3a8a', linewidth=2, markersize=4)# Línea de tendencia (regresión lineal simple)x_num = np.arange(len(df_ventas))coefs = np.polyfit(x_num, df_ventas['ventas_millones'], 1)ax.plot(df_ventas['mes'], np.polyval(coefs, x_num), '--', color='#dc2626',        linewidth=2, label=f'Tendencia: +{coefs[0]:.2f} millones/mes')ax.set_title('Ventas Mensuales 2022–2024: Tendencia Creciente con Estacionalidad',             fontweight='bold', fontsize=12)ax.set_xlabel('Mes')ax.set_ylabel('Ventas (millones $)')ax.legend()ax.spines['top'].set_visible(False)ax.spines['right'].set_visible(False)plt.tight_layout()plt.show()print(f"Crecimiento estimado: {coefs[0]:.2f} millones de dólares por mes.")print("Se observan picos recurrentes cada 12 meses — señal de estacionalidad (posiblemente fin de año).")

---## 3. Técnica 2 — Histograma: Distribución de Tiempos de Respuesta ⏱️Con 10,000 transacciones individuales, un histograma nos permite ver de inmediato si el sistema cumple el SLA de 300 ms y qué tan frecuentes son los incidentes de lentitud.

In [ ]:
# ============================================================# Técnica 2: Histograma + KDE de tiempos de respuesta# ============================================================fig, ax = plt.subplots(figsize=(10, 5))ax.hist(df_latencia['tiempo_respuesta_ms'], bins=60, color='#0ea5e9',        edgecolor='white', alpha=0.8, density=True)sns.kdeplot(df_latencia['tiempo_respuesta_ms'], ax=ax, color='#1e3a8a', linewidth=2)sla_ms = 300pct_incumple = (df_latencia['tiempo_respuesta_ms'] > sla_ms).mean() * 100ax.axvline(sla_ms, color='#dc2626', linestyle='--', linewidth=2,           label=f'SLA: {sla_ms} ms ({pct_incumple:.1f}% de transacciones lo incumplen)')ax.set_title('Distribución de Tiempos de Respuesta del Checkout', fontweight='bold', fontsize=12)ax.set_xlabel('Tiempo de respuesta (ms)')ax.set_ylabel('Densidad')ax.legend()ax.spines['top'].set_visible(False)ax.spines['right'].set_visible(False)plt.tight_layout()plt.show()print(f"Media: {df_latencia['tiempo_respuesta_ms'].mean():.0f} ms | Mediana: {df_latencia['tiempo_respuesta_ms'].median():.0f} ms")print(f"P95: {df_latencia['tiempo_respuesta_ms'].quantile(0.95):.0f} ms | P99: {df_latencia['tiempo_respuesta_ms'].quantile(0.99):.0f} ms")print(f"⚠️ {pct_incumple:.1f}% de las transacciones superan el SLA de {sla_ms} ms — hay una 'cola' de incidentes de lentitud.")

---## 4. Técnica 3 — Scatter Plot: ¿El Tráfico Explica la Lentitud? 🔗Sospechamos que los picos de lentitud coinciden con días de mayor tráfico. Un scatter plot con línea de tendencia nos permite confirmar (o descartar) esta hipótesis.

In [ ]:
# ============================================================# Técnica 3: Scatter plot tráfico vs latencia promedio diaria# ============================================================fig, ax = plt.subplots(figsize=(9, 5))ax.scatter(df_trafico['trafico_diario'], df_trafico['latencia_promedio_ms'],           color='#7c3aed', alpha=0.6, s=40, edgecolor='white')x_fit = df_trafico['trafico_diario'].valuesy_fit = df_trafico['latencia_promedio_ms'].valuescoefs = np.polyfit(x_fit, y_fit, 1)x_line = np.linspace(x_fit.min(), x_fit.max(), 100)ax.plot(x_line, np.polyval(coefs, x_line), '--', color='#dc2626', linewidth=2, label='Tendencia lineal')r = np.corrcoef(x_fit, y_fit)[0, 1]ax.set_title(f'Tráfico Diario vs Latencia Promedio | r = {r:.2f}', fontweight='bold', fontsize=12)ax.set_xlabel('Tráfico diario (visitas)')ax.set_ylabel('Latencia promedio (ms)')ax.legend()ax.spines['top'].set_visible(False)ax.spines['right'].set_visible(False)plt.tight_layout()plt.show()print(f"Correlación r = {r:.2f}: a mayor tráfico diario, mayor latencia promedio.")print("Hipótesis confirmada — el equipo de infraestructura debería escalar recursos en días de alto tráfico.")

---## 5. Técnica Complementaria — Box Plot: Latencia por Trimestre 📦Como cierre del caso, comparamos la latencia por trimestre usando box plots, útil para ver si los incidentes de lentitud se concentran en algún periodo particular (ej. temporadas de alto tráfico como fin de año).

In [ ]:
# ============================================================# Técnica complementaria: Box plot de latencia por trimestre simulado# ============================================================np.random.seed(11)n = len(df_latencia)trimestre = np.random.choice(['Q1', 'Q2', 'Q3', 'Q4'], size=n, p=[0.22, 0.22, 0.22, 0.34])df_latencia['trimestre'] = trimestre# Inyectar más lentitud en Q4 (temporada alta)mask_q4 = df_latencia['trimestre'] == 'Q4'df_latencia.loc[mask_q4, 'tiempo_respuesta_ms'] *= 1.35fig, ax = plt.subplots(figsize=(9, 5))sns.boxplot(data=df_latencia, x='trimestre', y='tiempo_respuesta_ms', ax=ax,            order=['Q1', 'Q2', 'Q3', 'Q4'], palette='Blues')ax.axhline(300, color='#dc2626', linestyle='--', linewidth=1.5, label='SLA: 300 ms')ax.set_title('Latencia del Checkout por Trimestre', fontweight='bold', fontsize=12)ax.set_xlabel('Trimestre')ax.set_ylabel('Tiempo de respuesta (ms)')ax.legend()ax.spines['top'].set_visible(False)ax.spines['right'].set_visible(False)plt.tight_layout()plt.show()for q in ['Q1', 'Q2', 'Q3', 'Q4']:    med = df_latencia[df_latencia['trimestre'] == q]['tiempo_respuesta_ms'].median()    print(f"{q}: mediana = {med:.0f} ms")print("\nQ4 muestra la mediana más alta y más outliers — coincide con temporada de alto tráfico (fin de año).")

---## Resumen y Puntos Clave 🎯- Este caso integró **cuatro técnicas** de visualización numérica sobre datos sintéticos realistas: line graph (tendencia de ventas), histograma+KDE (distribución de latencia), scatter plot (relación tráfico-latencia) y box plot (comparación por trimestre).- El flujo típico de un análisis numérico es: **generar/cargar datos → explorar distribución → buscar relaciones → segmentar y comparar → comunicar el hallazgo con la técnica adecuada**.- Combinar varias técnicas sobre el mismo problema (en vez de una sola) da una imagen mucho más completa: la línea mostró la tendencia, el histograma mostró el incumplimiento del SLA, el scatter explicó una causa probable, y el box plot ubicó el problema en el tiempo.- Este cierra la sección 2.3 del libro guía; el Módulo 06 continúa con técnicas de visualización aún más avanzadas (series de tiempo multivariadas, geoespacial, dashboards).

### 🧠 Autoevaluación1. En el caso del checkout, ¿por qué se usó un histograma en vez de un scatter plot para analizar los tiempos de respuesta?2. ¿Qué evidencia visual del caso soporta la hipótesis de que el tráfico alto causa mayor latencia?3. Si el equipo directivo te pidiera un solo gráfico para resumir "¿cumplimos el SLA por trimestre?", ¿cuál de las 4 técnicas usadas elegirías y por qué?4. Propón una quinta técnica (no usada en este caso) que podrías aplicar para enriquecer este análisis, y justifica tu elección.